# Appendix A — A Python Refresher for Electrochemical Simulation

*Python-native adaptation of Michael Honeychurch,
**Simulating Electrochemical Reactions in Mathematica** (SERM), Chapter 0
("Introduction to Mathematica"). The original chapter is a Wolfram-language
primer; this appendix re-casts the same material as a refresher in
numpy/scipy/matplotlib — the toolchain used throughout the rest of this book.*

Honeychurch's Chapter 0 walks the reader through the Mathematica idioms that
recur in every later chapter: numbers and exact arithmetic, assignments,
pattern-defined functions, **lists** and the `Table`/`Range`/`Map`/`Apply`
machinery for building and transforming them, symbolic algebra, plotting with
`ListPlot`/`Plot`/`ListPlot3D`, animation, and file I/O. None of that survives
verbatim in Python, but every construct has a clean, usually *more direct*,
counterpart. This appendix is organised as a **translation table you can run**:
each section states the Mathematica idiom, gives the Python equivalent, and
demonstrates it with a worked example whose output is printed or plotted.

Because this is a language refresher rather than a physics chapter, the
validation section (Section 12) does not compare against a closed-form
electrochemical result. Instead it follows the *self-consistency* strategy from
the authoring spec: a battery of `assert`s confirms that each idiom mapping
actually produces the value we claim it does — including a cross-check that a
hand-written procedural loop, a vectorised numpy expression, and a
`functools.reduce` all agree.

## The standard preamble

Every notebook in this book starts the same way: put the repository root on the
import path so the shared `serm` package is importable from `notebooks/`, then
import the numerical stack. In Mathematica the analogous step is loading add-on
packages with ``Needs["Graphics`Colors`"]`` or ``<<`` (`Get`); in Python it is
`import`. Where Mathematica autoloads everything into one global context, Python
keeps libraries namespaced (`np.`, `plt.`), which is why the same function name
can live in several modules without clashing.

In [1]:
import os, sys

# Make the project package importable when run from notebooks/.
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
import matplotlib.pyplot as plt

import serm
from serm import filters

print("numpy", np.__version__)
print("serm exports:", [x for x in dir(serm) if not x.startswith("_")][:8], "...")

numpy 2.4.6
serm exports: ['annotations', 'cottrell_current', 'dx_dimensionless', 'echem', 'electrode_current', 'explicit_solve', 'filters', 'grids'] ...


## 1. Numbers: exact vs. approximate

Mathematica distinguishes an *exact* rational `5/4` (infinite precision) from an
*approximate* real `1.25`, and converts between them with `N[...]`. Python draws
the same line, but the exact type lives in the standard library
`fractions.Fraction`, while ordinary `/` always yields a float. `Head[5]` /
`Head[5.0]` — which report the type of an object in Mathematica — map onto
Python's `type(...)`.

| Mathematica | Python |
|---|---|
| `5/4` (exact) | `Fraction(5, 4)` |
| `N[5/4]` | `float(Fraction(5, 4))` or `5/4` |
| `Head[5]`, `Head[5.0]` | `type(5)`, `type(5.0)` |
| `Sqrt[-256]` | `cmath.sqrt(-256)` or `np.sqrt(-256+0j)` |
| `Round`, `Floor`, `Ceiling` | `round`, `math.floor`, `math.ceil` |

In [2]:
from fractions import Fraction
import math, cmath

y = Fraction(5, 4)              # exact rational, like 5/4 in Mathematica
print("exact   :", y, "  type:", type(y).__name__)
print("N[5/4]  :", float(y))    # apply N[...]

print("Head[5], Head[5.0]:", type(5).__name__, type(5.0).__name__)
print("Sqrt[-256]:", cmath.sqrt(-256))          # 16j
print("Round/Floor/Ceiling[3.3]:",
      round(3.3), math.floor(3.3), math.ceil(3.3))

exact   : 5/4   type: Fraction
N[5/4]  : 1.25
Head[5], Head[5.0]: int float
Sqrt[-256]: 16j
Round/Floor/Ceiling[3.3]: 3 3 4


## 2. Assignments: immediate vs. delayed

Mathematica has *immediate* assignment `=` (`Set`, right-hand side evaluated
once, now) and *delayed* assignment `:=` (`SetDelayed`, re-evaluated every time
the symbol is used). The classic demonstration is `Random[]`: an immediate
`r1 = Random[]` freezes one number, whereas a delayed `r2 := Random[]` draws a
fresh number on each reference.

Python has only one binding operator, `=`, which is always immediate. To get the
"delayed", re-evaluate-on-use behaviour you bind a **function** (a `lambda` or
`def`) and call it. So Mathematica's `r2 := Random[]` becomes
`r2 = lambda: rng.random()`, evaluated as `r2()`.

In [3]:
rng = np.random.default_rng(0)        # seeded for reproducibility

r1 = rng.random()                     # immediate: one frozen value (Set, =)
r2 = lambda: rng.random()             # delayed:   re-evaluated on call (SetDelayed, :=)

print("immediate r1 (same every time):", [r1, r1, r1])
print("delayed   r2 (fresh each call):", [round(r2(), 4) for _ in range(3)])

immediate r1 (same every time): [0.6369616873214543, 0.6369616873214543, 0.6369616873214543]
delayed   r2 (fresh each call): [0.2698, 0.041, 0.0165]


## 3. Functions and pattern matching

A Mathematica function is defined with a *pattern* argument: `g[x_] := x^2`
matches any single expression and names it `x`. Restricted patterns add a type
or predicate: `h[x_Integer]` fires only on integers, `h2[x_Integer?Positive]`
only on positive integers. *Pure* functions are written `Function[x, x^2+3x]` or
in shorthand `(#^2 + 3#)&`.

Python expresses the plain case with `def` or `lambda`. The restricted patterns
have no single-line syntactic equivalent; the idiomatic translation is an
explicit guard (a conditional / `isinstance` test) inside the function, or
`functools.singledispatch` for genuine type-based dispatch. `(#^2+3#)&` is just
a `lambda`.

| Mathematica | Python |
|---|---|
| `g[x_] := x^2` | `def g(x): return x**2` |
| `(#^2 + 3#)&` | `lambda x: x**2 + 3*x` |
| `h[x_Integer] := x^2` | guard: `if isinstance(x, int): ...` |
| `h2[x_Integer?Positive]` | guard: `if isinstance(x, int) and x > 0` |

In [4]:
def g(x):
    'Square the argument -- the Mathematica g[x_] := x^2.'
    return x**2

cube = lambda x: x**3                      # pure function, (#^3)&

def h(x):
    'h[x_Integer] := x^2 -- only act on (Python) ints, else return unevaluated.'
    if isinstance(x, int):
        return x**2
    return ("h", x)                        # mimic Mathematica returning h[expr] unevaluated

print("g(7)      =", g(7))
print("cube(4)   =", cube(4))
print("h on [2, -3, 'a+b']:", [h(2), h(-3), h("a+b")])

g(7)      = 49
cube(4)   = 64
h on [2, -3, 'a+b']: [4, 9, ('h', 'a+b')]


## 4. Lists: the workhorse

Lists are where the Mathematica-to-Python mapping matters most, because every
simulation in this book is ultimately array arithmetic. Honeychurch stresses
that *vectorised* operations on whole lists are far more efficient than looping
element by element — exactly numpy's design philosophy.

A Mathematica `List` `{1,2,3,4,5,6}` maps to either a Python `list` or, when you
want elementwise math, a **numpy array**. The crucial point: arithmetic that
Mathematica threads automatically over a list (`2*mylist`, `mylist^2`,
`Log[mylist]`) is exactly what a numpy `ndarray` does, whereas a bare Python
`list` does *not* (`2*[1,2,3]` repeats the list). So the rule of thumb is: use
`np.array` whenever you want Mathematica-style listable arithmetic.

### Creating lists

| Mathematica | Python (numpy) |
|---|---|
| `Range[6]` | `np.arange(1, 7)` |
| `Range[7, 12]` | `np.arange(7, 13)` |
| `Table[Log[x], {x, 1, 6}]` | `np.log(np.arange(1, 7))` |
| `Table[expr, {i, a, b, d}]` | `np.arange(a, b+d, d)` then map |
| `Length[mylist]` | `len(arr)` or `arr.size` |

In [5]:
# Range[6] and Range[7,12].  Mathematica Range is 1-based and inclusive;
# np.arange is 0-based and half-open, so shift the bounds.
print("Range[6]    ->", np.arange(1, 7))
print("Range[7,12] ->", np.arange(7, 13))

mylist = np.array([1, 2, 3, 4, 5, 6])

# Listable arithmetic threads automatically over the whole array (no loop).
print("2*mylist    ->", 2 * mylist)
print("mylist^2    ->", mylist**2)
print("mylist+100  ->", mylist + 100)
print("Log[mylist] ->", np.round(np.log(mylist), 4))

# Table[Log[x], {x, 1, 6}] is just a listable function over a Range.
print("Table[Log]  ->", np.round(np.log(np.arange(1, 7)), 4))
print("Length      ->", mylist.size)

Range[6]    -> [1 2 3 4 5 6]
Range[7,12] -> [ 7  8  9 10 11 12]
2*mylist    -> [ 2  4  6  8 10 12]
mylist^2    -> [ 1  4  9 16 25 36]
mylist+100  -> [101 102 103 104 105 106]
Log[mylist] -> [0.     0.6931 1.0986 1.3863 1.6094 1.7918]
Table[Log]  -> [0.     0.6931 1.0986 1.3863 1.6094 1.7918]
Length      -> 6


### Modifying and combining lists

The Mathematica vocabulary `Reverse`, `Rest`, `Drop`, `Take`, `Append`/`Join`,
`Flatten`, `Sort`, `Union`, `Partition`, `Transpose` all have direct Python or
numpy counterparts. Slicing is where Python is much terser: `Drop[mylist, 1]` and
`Rest[mylist]` are both just `mylist[1:]`.

| Mathematica | Python |
|---|---|
| `Reverse[mylist]` | `mylist[::-1]` |
| `Rest[mylist]`, `Drop[mylist, 1]` | `mylist[1:]` |
| `Drop[mylist, {2, 4}]` | `np.delete(mylist, [1,2,3])` |
| `Take[mylist, {2, 3}]` | `mylist[1:3]` |
| `Join[a, b]` | `np.concatenate([a, b])` |
| `Flatten[m]` | `m.ravel()` / `np.ravel(m)` |
| `Sort`, `Union` | `np.sort`, `np.unique` |
| `Partition[list, 3]` | `arr.reshape(-1, 3)` |
| `Transpose[{a, b}]` | `np.column_stack([a, b])` / `arr.T` |

In [6]:
print("Reverse        ->", mylist[::-1])
print("Rest/Drop[,1]  ->", mylist[1:])
print("Drop[,{2,4}]   ->", np.delete(mylist, [1, 2, 3]))   # drop elements 2..4 (1-based)
print("Take[,{2,3}]   ->", mylist[1:3])                    # elements 2..3 (1-based)

a = np.array([1, 2, 3, 4])
b = np.array([5, 6, 7, 8])
print("Join[a,b]      ->", np.concatenate([a, b]))
print("Transpose      ->\n", np.column_stack([a, b]))      # pairs {a_i, b_i}

matrix = np.array([[1, 2, 3], [5, 4, 3], [5, 6, 7]])
flat = matrix.ravel()
print("Flatten        ->", flat)
print("Sort//Union    ->", np.unique(flat))                # Sort then Union (dedup)
print("Partition[,3]  ->\n", flat.reshape(-1, 3))

Reverse        -> [6 5 4 3 2 1]
Rest/Drop[,1]  -> [2 3 4 5 6]
Drop[,{2,4}]   -> [1 5 6]
Take[,{2,3}]   -> [2 3]
Join[a,b]      -> [1 2 3 4 5 6 7 8]
Transpose      ->
 [[1 5]
 [2 6]
 [3 7]
 [4 8]]
Flatten        -> [1 2 3 5 4 3 5 6 7]
Sort//Union    -> [1 2 3 4 5 6 7]
Partition[,3]  ->
 [[1 2 3]
 [5 4 3]
 [5 6 7]]


### Extracting elements: `Part` and `[[...]]`

Mathematica indexes with `Part` / `[[...]]` and is **1-based**; numpy is
**0-based**, so every index drops by one. Fancy indexing
`matrix[[{1,3},{1,3}]]` — pull rows 1 and 3, then columns 1 and 3 — becomes
`matrix[np.ix_([0,2], [0,2])]`. Boolean selection (Mathematica's `Cases` with a
predicate, or a `/;` condition) becomes a numpy boolean mask.

| Mathematica | Python |
|---|---|
| `First[v]`, `Last[v]` | `v[0]`, `v[-1]` |
| `matrix[[2]]` | `matrix[1]` |
| `matrix[[2, 3]]` | `matrix[1, 2]` |
| `matrix[[{1,3},{1,3}]]` | `matrix[np.ix_([0,2], [0,2])]` |
| `Min`, `Max` | `arr.min()`, `arr.max()` |

In [7]:
print("First, Last           :", flat[0], flat[-1])
print("matrix[[2]]           :", matrix[1])          # 1-based row 2 -> 0-based 1
print("matrix[[2,3]]         :", matrix[1, 2])       # element (row2, col3)
print("matrix[[{1,3},{1,3}]] :\n", matrix[np.ix_([0, 2], [0, 2])])  # corner 2x2
print("Min, Max              :", matrix.min(), matrix.max())

First, Last           : 1 7
matrix[[2]]           : [5 4 3]
matrix[[2,3]]         : 3
matrix[[{1,3},{1,3}]] :
 [[1 3]
 [5 7]]
Min, Max              : 1 7


## 5. Symbolic algebra: `Expand`, `Solve`, `/.`, `D`, `Integrate`

Mathematica is symbolic by default; Python is numeric by default and reaches for
**sympy** when genuine symbol manipulation is needed (and, per this book's
ground rules, *only* then — sympy is never the numerical engine). The mapping is
close to one-to-one:

| Mathematica | sympy |
|---|---|
| `Expand[...]`, `Factor[...]` | `sp.expand(...)`, `sp.factor(...)` |
| `Together`, `Apart`, `Cancel` | `sp.together`, `sp.apart`, `sp.cancel` |
| `expr /. x -> 2` (`ReplaceAll`) | `expr.subs(x, 2)` |
| `Solve[eqn == 0, x]` | `sp.solve(sp.Eq(lhs, 0), x)` |
| `D[f[x], x]` | `sp.diff(f, x)` |
| `Integrate[f, {x, 0, oo}]` | `sp.integrate(f, (x, 0, sp.oo))` |

A `/.` transformation rule (`Sqrt[x^2+y^2] /. x -> 2`) is sympy's `.subs`.

In [8]:
import sympy as sp

x, y, p, z = sp.symbols("x y p z")   # p is a free parameter (was Mathematica a)

print("Expand   :", sp.expand(3*(x + 4*z)*(3*x - 2*z)))
print("Factor   :", sp.factor(9*x**2 + 30*x*z - 24*z**2))
print("ReplaceAll (/.): ", sp.sqrt(x**2 + y**2).subs({x: 2, y: 2}))   # x->2, y->2

# Solve[x^3 - 4 x^2 + 2 a x == 0, x]  (parameter named p here to avoid clobbering)
roots = sp.solve(sp.Eq(x**3 - 4*x**2 + 2*p*x, 0), x)
print("Solve cubic :", roots)

# D[f[x], x] and Integrate[Exp[-x/b], {x, 0, Infinity}], Assumptions b>0
bpos = sp.symbols("b", positive=True)
print("D[x^3] :", sp.diff(x**3, x))
print("Integrate Exp[-x/b] over [0,oo) :",
      sp.integrate(sp.exp(-x/bpos), (x, 0, sp.oo)))

Expand   : 9*x**2 + 30*x*z - 24*z**2
Factor   : 3*(x + 4*z)*(3*x - 2*z)
ReplaceAll (/.):  2*sqrt(2)
Solve cubic : [0, -sqrt(2)*sqrt(2 - p) + 2, sqrt(2)*sqrt(2 - p) + 2]
D[x^3] : 3*x**2
Integrate Exp[-x/b] over [0,oo) : b


## 6. Programming styles: `Do`, `Map`, `Apply`

Honeychurch contrasts three ways to compute a factorial in Mathematica: a
procedural `Do` loop, a recursive pattern function, and the high-level
`Apply[Times, Range[n]]`. He notes that the high-level form, which operates on
the whole structure at once, is the most efficient. The same three styles exist
in Python, and the same advice holds — prefer the vectorised / whole-structure
form.

| Mathematica | Python |
|---|---|
| `Do[z = z n, {n, 1, 6}]` | `for n in range(1, 7): z *= n` |
| `Map[f, list]`, `f /@ list` | `[f(e) for e in list]` / `np.vectorize` |
| `Apply[Times, list]`, `Times @@ list` | `math.prod(list)` / `np.prod` |
| `Apply[Plus, list]`, `Plus @@ list` | `sum(list)` / `np.sum` |

`Apply` replaces the *head* of an expression — `Times @@ {a,b,c,d}` turns the
list into a product. In Python the closest concept is `functools.reduce`, but
for the common heads (`Plus`, `Times`) the dedicated builtins `sum`/`math.prod`
are clearer and faster.

In [9]:
import math
from functools import reduce
import operator

# Three factorials, three styles -- all must give 6! = 720.
def fac_procedural(n):
    'Do[z = z n, {n, 1, m}] -- procedural loop.'
    z = 1
    for k in range(1, n + 1):
        z *= k
    return z

def fac_recursive(n):
    'fac[n_] := n fac[n-1]; fac[0] = 1 -- recursive pattern function.'
    return 1 if n == 0 else n * fac_recursive(n - 1)

def fac_apply(n):
    'Apply[Times, Range[n]] -- high-level, whole-structure.'
    return math.prod(range(1, n + 1))

print("procedural :", fac_procedural(6))
print("recursive  :", fac_recursive(6))
print("Apply/prod :", fac_apply(6))

# Map (f /@ list) and Apply (Plus @@, Times @@) on a small list.
data = [1, 2, 3, 4]
print("Map sqrt (/@)   :", [round(math.sqrt(e), 4) for e in data])
print("Apply[Plus] (@@):", sum(data), "==", reduce(operator.add, data))
print("Apply[Times](@@):", math.prod(data), "==", reduce(operator.mul, data))

procedural : 720
recursive  : 720
Apply/prod : 720
Map sqrt (/@)   : [1.0, 1.4142, 1.7321, 2.0]
Apply[Plus] (@@): 10 == 10
Apply[Times](@@): 24 == 24


## 7. Plotting in 2-D: `Plot` and `ListPlot`

Mathematica's `Plot[f, {x, a, b}]` samples a function over a range and joins the
points; `ListPlot[data]` plots discrete `{x, y}` pairs. Both collapse to
matplotlib: sample the function on a numpy grid, then `plt.plot` (joined) or
`plt.scatter` / `plt.plot(..., "o")` (points). The cached Wolfram graphics from
the source notebook are never reused — every figure here is regenerated.

| Mathematica | matplotlib |
|---|---|
| `Plot[Sin[x], {x, 0, 2 Pi}]` | `plt.plot(x, np.sin(x))` over a grid |
| `ListPlot[pairs]` | `plt.plot(xs, ys, "o")` |
| `Show[p1, p2]` | call `plt.plot` twice on one axes |
| `PlotStyle -> {Red, Dashing[...]}` | `color=`, `linestyle=` kwargs |

In [10]:
fig, ax = plt.subplots(figsize=(5.5, 3.6))

# Plot[Sin[x], {x, 0, 2 Pi}] -- a dense grid joined into a smooth curve.
xs = np.linspace(0, 2 * np.pi, 400)
ax.plot(xs, np.sin(xs), color="C3", lw=1.5, label="Plot: sin(x)")

# ListPlot[Table[{x, Sin[x]}, {x, 0, 2 Pi, Pi/10}]] -- discrete sampled points.
xd = np.arange(0, 2 * np.pi + 1e-9, np.pi / 10)
ax.plot(xd, np.sin(xd), "o", color="C0", ms=4, label="ListPlot: sampled")

# Show[...] of a second curve overlaid on the same axes.
ax.plot(xs, np.cos(xs), "--", color="0.4", lw=1.2, label="Show: cos(x)")

ax.set_xlabel("x"); ax.set_ylabel("y"); ax.legend(frameon=False)
ax.set_title("Plot / ListPlot / Show  ->  matplotlib")
plt.tight_layout(); plt.show()

/tmp/ipykernel_1429538/2956185631.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 8. Plotting in 3-D: `ListPlot3D` and `Plot3D`

`ListPlot3D` renders a grid of $z$ values as a surface — exactly the shape of
the concentration arrays `c[x, t]` produced by the finite-difference solvers in
later chapters. The Python equivalent is matplotlib's `plot_surface` on a
`meshgrid`. The `serm.plotting` module wraps this as `plot_surface`, but to keep
this appendix self-contained we show the raw call too.

In [11]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers '3d' projection)

# Plot3D[Sin[r]/r style bump as a stand-in concentration surface z[x, t].
xg = np.linspace(-3, 3, 60)
tg = np.linspace(-3, 3, 60)
X, T = np.meshgrid(xg, tg)
R = np.sqrt(X**2 + T**2) + 1e-9
Z = np.sin(R) / R

fig = plt.figure(figsize=(5.5, 4))
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(X, T, Z, cmap="viridis", linewidth=0, antialiased=True)
ax.set_xlabel("x"); ax.set_ylabel("t"); ax.set_zlabel("z")
ax.set_title("ListPlot3D / Plot3D  ->  plot_surface")
plt.tight_layout(); plt.show()

/tmp/ipykernel_1429538/2901422681.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 9. Animation: `Table[Plot[...]]` and `Animate`

Honeychurch animates a sine wave whose frequency sweeps by generating a `Table`
of `Plot`s (or with `Animate`/`MoviePlot`) and flipping through the frames. The
matplotlib counterpart is `FuncAnimation`. For a headless-executed notebook the
animation has to be embedded as HTML rather than shown in a GUI window, which we
do with `HTML(anim.to_jshtml())` — the same approach `serm.plotting.animate_*`
uses. We keep the frame count small so the notebook executes quickly.

In [12]:
import matplotlib.animation as animation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(5.0, 3.0))
xline = np.linspace(0, 10, 300)
(line,) = ax.plot(xline, np.sin(xline), color="C3")
ax.set_ylim(-1.1, 1.1); ax.set_xlabel("x")
ax.set_title("Animate[Plot[Sin[a x]]]  ->  FuncAnimation")

freqs = np.linspace(1.0, 5.0, 9)     # Animate[..., {a, 1, 5}]

def update(a):
    line.set_ydata(np.sin(a * xline))
    return (line,)

anim = animation.FuncAnimation(fig, update, frames=freqs, interval=200, blit=True)
plt.close(fig)                       # don't also emit a static frame
HTML(anim.to_jshtml())

## 10. File I/O: `Export` / `Import` / `ReadList`

The simulations export two-column `{potential, current}` or `{x, c}` data.
Mathematica writes these with `Export["file.dat", data, "Table"]` and reads them
with `Import` / `ReadList`. In Python the direct equivalents are
`np.savetxt` and `np.loadtxt` (or pandas for labelled tables). We round-trip a
small array through a temp file to show the data survives intact.

| Mathematica | Python |
|---|---|
| `Export["f.dat", data, "Table"]` | `np.savetxt("f.dat", data)` |
| `Import["f.dat"]`, `ReadList` | `np.loadtxt("f.dat")` |
| `Directory[]` | `os.getcwd()` |

In [13]:
import tempfile

cv_data = np.column_stack([np.linspace(-0.3, 0.3, 7),       # potential
                           np.sin(np.linspace(-0.3, 0.3, 7))])  # mock current

with tempfile.TemporaryDirectory() as d:
    path = os.path.join(d, "simulatedCV.dat")
    np.savetxt(path, cv_data)                # Export[..., "Table"]
    reloaded = np.loadtxt(path)              # Import / ReadList
    print("round-trip max abs error:", np.max(np.abs(cv_data - reloaded)))

round-trip max abs error: 0.0


## 11. Putting it together with `serm`

The whole point of this translation table is that later chapters lean on these
idioms through the shared `serm` package. As a concrete taste, here is a noisy
signal smoothed with `serm.filters.moving_average` — the Python port of the
book's `Filters.m` — exactly the kind of list operation (a windowed average over
a `Table` of points) that Chapter 0 motivates. We plot raw vs. smoothed, the
`Show`-style overlay from Section 7.

In [14]:
rng2 = np.random.default_rng(42)
clean = np.sin(np.linspace(0, 4 * np.pi, 200))
noisy = clean + 0.3 * rng2.standard_normal(clean.shape)
win = 9
smoothed = filters.moving_average(noisy, win)   # 'valid' mode: shorter by win-1
# 'valid'-mode correlation drops (win-1)/2 points from each end; line the
# smoothed output up against the matching centred slice of the originals.
half = (win - 1) // 2
idx = np.arange(half, half + smoothed.size)     # x-positions of the smoothed points
clean_c = clean[half:half + smoothed.size]
noisy_c = noisy[half:half + smoothed.size]

fig, ax = plt.subplots(figsize=(5.5, 3.4))
ax.plot(noisy, color="0.7", lw=0.8, label="noisy")
ax.plot(idx, smoothed, color="C3", lw=1.6, label=f"moving_average({win})")
ax.plot(clean, color="C0", lw=1.0, ls="--", label="true signal")
ax.set_xlabel("sample"); ax.legend(frameon=False)
ax.set_title("serm.filters.moving_average on a noisy Table")
plt.tight_layout(); plt.show()

# Smoothing must reduce the RMS distance to the underlying clean signal.
rms_noisy = np.sqrt(np.mean((noisy_c - clean_c) ** 2))
rms_smooth = np.sqrt(np.mean((smoothed - clean_c) ** 2))
print(f"RMS error  noisy: {rms_noisy:.4f}   smoothed: {rms_smooth:.4f}")

RMS error  noisy: 0.2667   smoothed: 0.0784


/tmp/ipykernel_1429538/56000160.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 12. Validation

This is a language appendix, not a physics chapter, so there is no
electrochemical closed form to compare against. Following the authoring spec's
*self-consistency* policy (stated explicitly here), validation instead asserts
that **every idiom mapping above produces the value we claim**, and that the
three programming styles agree. Concretely:

1. The numpy list operations reproduce the Mathematica results
   (`Range`, listable arithmetic, `Reverse`, `Partition`, `Transpose`,
   corner-extraction with `np.ix_`).
2. The symbolic results match hand-computed values
   (`Solve[x^3-4x^2+2ax]` has roots $\{0, 2-\sqrt{4-2a},\,2+\sqrt{4-2a}\}$;
   $\int_0^\infty e^{-x/b}\,dx = b$).
3. Procedural, recursive, and high-level factorials all equal `math.factorial(6)`.
4. `Map`/`Apply` reproductions equal numpy's reductions.
5. The file round-trip is loss-free and `moving_average` reduces the RMS error.

Each check raises `AssertionError` on failure; a clean run prints `PASS`.

In [15]:
# --- 1. list operations ----------------------------------------------------
assert np.array_equal(np.arange(1, 7), [1, 2, 3, 4, 5, 6])
assert np.array_equal(2 * mylist, [2, 4, 6, 8, 10, 12])
assert np.array_equal(mylist[::-1], [6, 5, 4, 3, 2, 1])
assert np.allclose(np.log(np.arange(1, 7)), [math.log(k) for k in range(1, 7)])
assert np.array_equal(flat.reshape(-1, 3), [[1, 2, 3], [5, 4, 3], [5, 6, 7]])
assert np.array_equal(np.column_stack([a, b]),
                      [[1, 5], [2, 6], [3, 7], [4, 8]])         # Transpose[{a,b}]
assert np.array_equal(matrix[np.ix_([0, 2], [0, 2])], [[1, 3], [5, 7]])  # corners

# --- 2. symbolic results ---------------------------------------------------
expected_roots = {sp.Integer(0),
                  2 - sp.sqrt(4 - 2*p),
                  2 + sp.sqrt(4 - 2*p)}
got_roots = set(sp.solve(sp.Eq(x**3 - 4*x**2 + 2*p*x, 0), x))
assert sp.simplify(sum(got_roots) - sum(expected_roots)) == 0   # Vieta: roots sum to 4
# Match elementwise up to symbolic form (sqrt(4-2p) vs sqrt(2) sqrt(2-p) etc.).
assert all(any(sp.simplify(g - e) == 0 for e in expected_roots) for g in got_roots)
assert sp.integrate(sp.exp(-x/bpos), (x, 0, sp.oo)) == bpos     # == b

# --- 3. three factorial styles agree --------------------------------------
ref = math.factorial(6)
assert fac_procedural(6) == fac_recursive(6) == fac_apply(6) == ref == 720

# --- 4. Map / Apply reproductions -----------------------------------------
assert math.prod(data) == reduce(operator.mul, data) == np.prod(data)
assert sum(data) == reduce(operator.add, data) == np.sum(data)

# --- 5. file round-trip + smoothing ---------------------------------------
assert np.max(np.abs(cv_data - reloaded)) == 0.0
assert rms_smooth < rms_noisy            # smoothing genuinely helps

print("PASS: all", "idiom mappings reproduce their claimed values.")
print(f"  cubic root sum (Vieta) = {sum(got_roots)}  (expect 4)")
print(f"  factorial(6) = {ref}")
print(f"  RMS error noisy {rms_noisy:.4f} -> smoothed {rms_smooth:.4f}")

PASS: all idiom mappings reproduce their claimed values.
  cubic root sum (Vieta) = 4  (expect 4)
  factorial(6) = 720
  RMS error noisy 0.2667 -> smoothed 0.0784


## Summary

This appendix mapped the Mathematica idioms of Honeychurch's Chapter 0 onto
their idiomatic Python equivalents:

- **Numbers** — `Fraction` for exact rationals, `float`/numpy for approximate.
- **Assignments** — `=` is always immediate; delayed `:=` becomes a callable.
- **Functions / patterns** — `def`/`lambda`, with explicit guards replacing
  restricted patterns like `x_Integer?Positive`.
- **Lists** — numpy arrays give Mathematica's listable arithmetic; `Range`,
  `Table`, `Part`, `Map`, `Apply`, `Partition`, `Transpose` all have direct
  counterparts, remembering numpy is **0-based and half-open**.
- **Symbolic algebra** — sympy mirrors `Expand`/`Solve`/`/.`/`D`/`Integrate`,
  used only for genuinely symbolic work.
- **Plotting** — `Plot`/`ListPlot`/`ListPlot3D`/`Animate` all become matplotlib
  (`plt.plot`, `plot_surface`, `FuncAnimation`), regenerated from scratch.
- **File I/O** — `np.savetxt`/`np.loadtxt` stand in for `Export`/`Import`.

These are the building blocks every later chapter uses through the shared
`serm` package. The finite-difference solver of Chapter 2
(`notebooks/02_explicit_finite_differences.ipynb`) is where they first combine
into a real electrochemical simulation: a `make_grid` array stepped forward by
vectorised stencil arithmetic and plotted as concentration profiles and a
surface — every idiom in this appendix at work.